建立一个文件夹games
里面建立文件夹game,mysql和文件docker-compose.yml

In [ ]:
#在mysql中建立init.sql


CREATE DATABASE IF NOT EXISTS game_db;
USE game_db;

CREATE TABLE IF NOT EXISTS players (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50),
    guess INT,
    secret INT,
    is_win TINYINT(1),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

In [ ]:
#编写docker-compose.yml
version: "3.9"

services:
  mysql:
    image: mysql:8.0
    container_name: mysql-game
    restart: unless-stopped
    environment:
      MYSQL_ROOT_PASSWORD: rootpass
      MYSQL_DATABASE: game_db
    volumes:
      - ./mysql/init.sql:/docker-entrypoint-initdb.d/init.sql:ro
    ports:
      - "3306:3306"
    healthcheck:
      test: ["CMD", "mysqladmin", "ping", "-h", "localhost", "-prootpass"]
      interval: 5s
      timeout: 3s
      retries: 10

  game:
    build: ./game
    container_name: game-app
    restart: unless-stopped
    depends_on:
      mysql:
        condition: service_healthy
    ports:
      - "8000:8000"

在game文件夹中建立
requirements.txt
Dockerfile
main.py

In [ ]:
#编写main.py
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
import mysql.connector
import duckdb
import random
import time

app = FastAPI()

DB = {
    "host": "mysql",
    "port": 3306,
    "user": "root",
    "password": "rootpass",
    "database": "game_db",
}

# ---------- 获取玩家当前秘密数字 ----------
def get_secret(name: str) -> int:
    conn = mysql.connector.connect(**DB)
    cur = conn.cursor()
    
    # 查找未完成的游戏
    cur.execute(
        "SELECT secret FROM players WHERE name=%s AND is_win=0 ORDER BY id DESC LIMIT 1",
        (name,)
    )
    row = cur.fetchone()
    
    if row:
        secret = row[0]
    else:
        # 新游戏，生成新数字
        secret = random.randint(1, 100)
    
    conn.close()
    return secret

# ---------- 游戏接口 ----------
@app.post("/api/guess")
async def guess(request: Request):
    data = await request.json()
    name = str(data.get("name", "匿名")).strip()
    number = int(data.get("number"))

    secret = get_secret(name)
    is_win = 1 if number == secret else 0

    conn = mysql.connector.connect(**DB)
    cur = conn.cursor()

    # 记录本次猜测（不再插入 guess=0）
    cur.execute(
        "INSERT INTO players (name, guess, secret, is_win) VALUES (%s, %s, %s, %s)",
        (name, number, secret, is_win)
    )

    # 如果猜对了，标记所有未完成的对局为已完成
    if is_win:
        cur.execute(
            "UPDATE players SET is_win=1 WHERE name=%s AND is_win=0",
            (name,)
        )

    conn.commit()
    conn.close()

    return {
        "name": name,
        "guess": number,
        "secret": secret,
        "result": "🎉 猜对了！" if is_win else "❌ 猜错了",
        "finished": bool(is_win)
    }

# ---------- 排行榜 ----------
@app.get("/api/rank")
def rank():
    conn = mysql.connector.connect(**DB)
    cur = conn.cursor(dictionary=True)
    cur.execute("""
        SELECT name, 
               COUNT(DISTINCT CASE WHEN is_win=1 THEN id END) AS wins,
               COUNT(*) AS total_games
        FROM players 
        GROUP BY name 
        ORDER BY wins DESC
    """)
    rows = cur.fetchall()
    conn.close()
    return rows

# ---------- MCP 分析（修复版） ----------
@app.get("/mcp/analyze")
def mcp_analyze():
    try:
        conn = mysql.connector.connect(**DB)
        cur = conn.cursor()
        cur.execute("SELECT name, guess, secret, is_win FROM players WHERE guess > 0")
        rows = cur.fetchall()
        conn.close()

        if not rows:
            return {"msg": "暂无数据，先玩几局"}

        con = duckdb.connect()
        con.execute("""
            CREATE TABLE p (
                name VARCHAR,
                guess INT,
                secret INT,
                is_win INT
            )
        """)

        for r in rows:
            con.execute(
                "INSERT INTO p VALUES (?, ?, ?, ?)",
                [str(r[0]), int(r[1]), int(r[2]), int(r[3])]
            )

        top = con.execute("""
            SELECT name, SUM(is_win) AS wins
            FROM p GROUP BY name ORDER BY wins DESC LIMIT 1
        """).fetchone()

        hot = con.execute("""
            SELECT guess, COUNT(*) AS cnt
            FROM p GROUP BY guess ORDER BY cnt DESC LIMIT 3
        """).fetchall()

        return {
            "总游戏次数": len(rows),
            "最强玩家": {"姓名": top[0], "获胜次数": int(top[1])} if top else "暂无",
            "热门数字": [{"数字": h[0], "次数": h[1]} for h in hot],
            "MCP状态": "✅ DuckDB 分析成功",
        }

    except Exception as e:
        return {"error": str(e)}, 500

# ---------- 网页 ----------
@app.get("/", response_class=HTMLResponse)
def index():
    return """
<!DOCTYPE html>
<html lang="zh">
<head>
<meta charset="UTF-8"/>
<title>🎮 猜数字 · MCP</title>
<style>
 body{font-family:Segoe UI;background:#0f172a;color:#e2e8f0;display:flex;justify-content:center;align-items:center;height:100vh}
 .card{background:#1e293b;padding:30px;border-radius:14px;width:380px;text-align:center}
 input{padding:10px;width:100%;border-radius:8px;border:none;margin-bottom:10px;background:#334155;color:#fff}
 button{padding:10px;width:100%;border:none;border-radius:8px;background:#6366f1;color:#fff;font-weight:bold;cursor:pointer}
 .log{margin-top:15px;background:#0f172a;padding:12px;border-radius:10px;text-align:left;font-size:13px;white-space:pre-line}
 .badge{display:inline-block;padding:2px 10px;border-radius:999px;font-size:12px;margin-bottom:12px}
 .win{background:#166534;color:#bbf7d0}
 .lose{background:#7f1d1d;color:#fecaca}
</style>
</head>
<body>
<div class="card">
  <h1>🔮 猜数字</h1>
  <p>我心里想了一个 <b>1~100</b> 的数字</p>
  <div class="badge lose">每个玩家数字不同</div>
  <input id="name" placeholder="你的名字"/>
  <input id="num" type="number" placeholder="输入 1~100"/>
  <button onclick="guess()">🎯 提交</button>
  <div class="log" id="log"></div>
</div>

<script>
let currentSecret = null;

async function guess(){
  const name = document.getElementById("name").value.trim();
  const num = Number(document.getElementById("num").value);
  const log = document.getElementById("log");

  if(!name) return alert("请输入名字");
  if(!num || num<1 || num>100) return alert("请输入 1~100 的数字");

  const res = await fetch("/api/guess",{
    method:"POST",
    headers:{"Content-Type":"application/json"},
    body:JSON.stringify({name,number:num})
  });

  const d = await res.json();
  currentSecret = d.secret;

  const cls = d.finished ? "win" : "lose";
  log.innerHTML += `<span class="badge ${cls}">${d.result}</span>\n你猜了 <b>${d.guess}</b>，秘密数字是 <b>${d.secret}</b>\n`;
  log.scrollTop = log.scrollHeight;

  if(d.finished){
    log.innerHTML += `\n🎉 恭喜通关！刷新页面可以开始新游戏。\n`;
  }
}
</script>
</body>
</html>
"""

In [ ]:
#编写requirement.txt
fastapi==0.110.0
uvicorn[standard]==0.27.1
mysql-connector-python==8.3.0
duckdb==0.10.2

编写Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONUNBUFFERED=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

编写完成后打开powershell
cd C:\User\wjh\games
docker compose up -d --build


打开 http://localhost:8000
输入新名字，猜几次
访问 http://localhost:8000/api/rank
访问 http://localhost:8000/mcp/analyze